# Derived research notebook: historical factor-weight audit

**Purpose:** read-only comparison of the historical Laguna de Bay notebook vector and the *paper-weight sensitivity vector encoded in draft research code*. This is **not** proof of manuscript AHP/WorldPop reproduction, geographic validity, field validation or a quantum result.

The 25-cell original notebook and original GeoJSON are immutable historical inputs and **must not be overwritten**. Run this notebook from a checkout containing `scripts/verify_research_sources.py` (source provenance draft PR #159). Python 3.12 standard library only.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in (cwd, *cwd.parents) if (p / 'docs/research_protocols/original_water_sources_v1.json').is_file()), None)
if project_root is None:
    raise RuntimeError('Run from within a checkout with the frozen research-source catalog')
sys.path.insert(0, str(project_root))
from scripts.verify_research_sources import verify
verified_sources = verify(root=project_root, catalog_path=project_root / 'docs/research_protocols/original_water_sources_v1.json')
assert verified_sources['original_source_bytes_verified'] is True
print('Original notebook and GeoJSON bytes match the checked-in provenance catalog.')


## Two historically distinct weighting assumptions

The `NOTEBOOK_WEIGHTS` vector and alternative `PAPER_WEIGHTS` sensitivity vector are recorded in `problems/water_quality/notebook_reference_model.py` on the existing **draft scientific stack** (#155–157); the original notebook's Cell 2 is the historical baseline. We freeze a copy here for a **no-solver comparison**, not as a claim that a manuscript dataset or analytic hierarchy process has been independently reproduced.

**D2 in the historical GeoJSON is road proximity (`d2_road_proximity`)**, not independently verified WorldPop population exposure. Only identical factor labels, units and data can justify a direct modeled-outcome comparison; changing weights alone cannot establish paper parity.
The notebook now **parses historical CELL 2 using Python's AST, without executing the original notebook or importing NumPy**. The explicitly copied historical vector is only a regression canary against that independently extracted source input. The alternative vector remains a copied reference from draft research code, not separately published data custody.

In [ ]:
from decimal import Decimal
from scripts.historical_weight_contract import extract_original_weights

# Read the actual original CELL 2 Python AST without executing legacy code.
# The literal expected vector below is a regression canary, not the input source.
FACTORS, NOTEBOOK_WEIGHTS = extract_original_weights(project_root / 'problems/water_quality/notebooks/experiment9 grid search copy.ipynb')
EXPECTED_NOTEBOOK_WEIGHTS = tuple(map(Decimal, ('0.3671', '0.1224', '0.2508', '0.0502', '0.1278', '0.0183', '0.0476', '0.0159')))
assert NOTEBOOK_WEIGHTS == EXPECTED_NOTEBOOK_WEIGHTS, 'Historical weights no longer match the frozen CELL 2 canary'
PAPER_SENSITIVITY_WEIGHTS = tuple(map(Decimal, ('0.276095907', '0.276095907', '0.179220150', '0.089610075', '0.087188181', '0.043594091', '0.024097845', '0.024097845')))
assert len(FACTORS) == len(NOTEBOOK_WEIGHTS) == len(PAPER_SENSITIVITY_WEIGHTS) == 8
assert FACTORS[-1] == 'd2_road_proximity'


In [ ]:
print(f"{'factor':30} {'notebook':>12} {'sensitivity':>14} {'difference':>13}")
for factor, baseline, alternate in zip(FACTORS, NOTEBOOK_WEIGHTS, PAPER_SENSITIVITY_WEIGHTS):
    print(f'{factor:30} {str(baseline):>12} {str(alternate):>14} {str(baseline-alternate):>13}')
print('Historical notebook weight sum:', sum(NOTEBOOK_WEIGHTS))
print('Encoded sensitivity weight sum:', sum(PAPER_SENSITIVITY_WEIGHTS))


## Interpretation and reproducibility boundary

The historical notebook vector sums to **1.0001** (rounding); the encoded alternate vector sums to **1.000000001**. The mismatch is an **input parameter difference**, not a solver failure or evidence that one model is correct. No original values are silently renormalized in the table.

This notebook intentionally has **no stochastic sampler**, solver invocation, emitted QUBO/LP, network calls, plots that imply new data, or site-placement claims. Scientific model comparisons require the independently checked N20 input and objective/constraint evidence. See `docs/NOTEBOOK_DASHBOARD_PARITY_2026_10_26.md` for the full historical cell map.

In [ ]:
assert sum(NOTEBOOK_WEIGHTS) == Decimal('1.0001')
assert sum(PAPER_SENSITIVITY_WEIGHTS) == Decimal('1.000000001')
assert any(a != b for a, b in zip(NOTEBOOK_WEIGHTS, PAPER_SENSITIVITY_WEIGHTS))
assert verified_sources['paper_or_worldpop_parity_proven'] is False
print('Input-vector audit passed; manuscript parity remains unverified.')


## For the next research milestone

Add **separate, source-reviewed** N20 scientific model execution only after the input dataset, index mapping, objective, feasible set, compiler/LP/QUBO, numerical tolerance, and independent solution verifier are pinned. Keep original notebook code and GeoJSON unchanged; attach run provenance, solver versions, precision environment and expected outputs to any future derived executable notebook. Do not substitute this input comparison for the thesis result.